# DFIR-Guardrail: hard dataset eval (phi3:mini)
Runs the pipeline on `hard.json` (500 labelled artifacts: false-positive traps + evasive injections). Run all cells; upload `hard_eval_bundle.zip` when prompted; download `guardrail_simulation.json` at the end and send it back.

In [ ]:
# 1. Install Ollama and start the daemon
!sudo apt-get update -y
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time, requests
os.environ["OLLAMA_NUM_PARALLEL"] = "3"
ollama_process = subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
for i in range(30):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("Ollama daemon is ready!"); break
    except Exception:
        time.sleep(1)
else:
    print("Warning: Ollama daemon did not become ready within 30 seconds.")
!/usr/local/bin/ollama pull phi3:mini

In [ ]:
# 2. Clone repo, install deps, then overlay the exact local src/ + hard.json from the bundle
!git clone https://github.com/YedidyaBarGad/DFIR-Guardrail.git
%cd DFIR-Guardrail
!pip install -r requirements.txt

from google.colab import files
up = files.upload()  # choose hard_eval_bundle.zip
!unzip -o hard_eval_bundle.zip

In [ ]:
# 3. Two runs on the hard dataset (num_samples is ignored when --dataset_file is given)
# A) full pipeline: heuristic/semantic pre-filter + phi3
!mkdir -p out_pipeline out_modelonly
!PYTHONPATH=. python src/pipeline.py --mode sim --dataset_file hard.json --concurrency 3 --output_dir out_pipeline
# B) model only: every artifact goes to phi3 (comparable with the Haiku run)
!PYTHONPATH=. python src/pipeline.py --mode sim --dataset_file hard.json --concurrency 3 --no_prefilter --output_dir out_modelonly

In [ ]:
# 4. Download both reports and stop Ollama
import shutil
shutil.copy("out_pipeline/guardrail_simulation.json", "phi3_pipeline.json")
shutil.copy("out_modelonly/guardrail_simulation.json", "phi3_modelonly.json")
files.download("phi3_pipeline.json"); files.download("phi3_modelonly.json")
ollama_process.terminate()